In [3]:
# %pip install rich smdebug_rulesconfig schema
%pip freeze | grep -E "rich|smdebug|schema|sage"

fastjsonschema==2.21.2
jsonschema==4.26.0
jsonschema-specifications==2025.9.1
rich==14.3.3
sagemaker==2.235.2
sagemaker-core==1.0.76
sagemaker-experiments==0.1.45
sagemaker_training==4.9.0
schema==0.7.8
smdebug-rulesconfig==1.0.1
Note: you may need to restart the kernel to use updated packages.


In [4]:
from sagemaker.estimator import Estimator
from sagemaker.session import Session
from sagemaker import get_execution_role

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [6]:
import boto3
region_name = boto3.session.Session().region_name

sm_uri = f"155954279556.dkr.ecr.{region_name}.amazonaws.com/gs-automl-base-containers/boilerplate312_sm:hjsong-v1.0"
base_job_name='hjsong-training'

In [15]:
role = get_execution_role()
sess = Session()
print(role)

arn:aws:iam::155954279556:role/service-role/AmazonSageMaker-ExecutionRole-20260211T204344


In [9]:
estimator = Estimator(
    image_uri=sm_uri,
    role=role,
    instance_count=1,
    instance_type='ml.m5.large',
    hyperparameters={
        "table_name": "automl-classification-experiment",
        "project_hashkey": "2ee07a49",
        "experiment_hashkey": "1cbd8309",
        "dataset_table_name": "automl-dataset",
        "dataset_profile_table_name": "automl-dataset-profile-experiment-result",
        "model_repo_table_name": "automl-model-repo",
        "model_experiment_result_table_name": "automl-classification-experiment",
        "username": "sean@gs.co.kr",
        "job_type": "training",
        "task_token": "1234",
    },
    base_job_name=base_job_name,
    sagemaker_session=sess,
    # 태그 설정 (SCP 요구사항 충족)
    tags=[
        {'Key': 'Environment', 'Value': 'dev'},
        {'Key': 'Project', 'Value': 'automl'},
        {'Key': 'Owner', 'Value': 'sean'},
        {'Key': 'CostCenter', 'Value': 'gs-retail'}
    ],
    # 기존 버킷 사용 (버킷 생성 방지)
    output_path=f's3://retail-mlops-edu-202602/output',
)

try:
    estimator.fit()
except ValueError as e:
    print(e)

INFO:sagemaker:Creating training-job with name: sean-training-2026-03-06-01-49-19-312


2026-03-06 01:49:20 Starting - Starting the training job...
2026-03-06 01:49:35 Starting - Preparing the instances for training...
2026-03-06 01:50:25 Downloading - Downloading the training image.....2026-03-06 01:51:03,897 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-03-06 01:51:03,898 sagemaker-training-toolkit INFO     Failed to parse hyperparameter dataset_profile_table_name value automl-dataset-profile-experiment-result to Json.
Returning the value itself
2026-03-06 01:51:03,898 sagemaker-training-toolkit INFO     Failed to parse hyperparameter dataset_table_name value automl-dataset to Json.
Returning the value itself
2026-03-06 01:51:03,898 sagemaker-training-toolkit INFO     Failed to parse hyperparameter experiment_hashkey value 1cbd8309 to Json.
Returning the value itself
2026-03-06 01:51:03,898 sagemaker-training-toolkit INFO     Failed to parse hyperparameter job_type value training to Json.
Returning the value itself
2026-03-06 01

In [11]:
import boto3
import time

def tail_training_logs(training_job_name, log_stream_name, region="us-east-1"):
    logs = boto3.client("logs", region_name=region)
    sm = boto3.client("sagemaker", region_name=region)
    
    log_group = "/aws/sagemaker/TrainingJobs"
    next_token = None
    terminal_states = {"Completed", "Failed", "Stopped"}

    while True:
        kwargs = {
            "logGroupName": log_group,
            "logStreamName": log_stream_name,
            "startFromHead": True,
        }
        if next_token:
            kwargs["nextToken"] = next_token

        try:
            response = logs.get_log_events(**kwargs)
            for event in response["events"]:
                print(event["message"])
            next_token = response["nextForwardToken"]
        except logs.exceptions.ResourceNotFoundException:
            print("로그 스트림 아직 생성 안됨, 대기 중...")

        # exit 조건: job 상태 확인
        job_status = sm.describe_training_job(
            TrainingJobName=training_job_name
        )["TrainingJobStatus"]

        if job_status in terminal_states:
            print(f"\n✅ Training job 종료: {job_status}")
            break

        time.sleep(2)

In [12]:
import boto3

session = boto3.session.Session()
region_name = session.region_name

training_job_name = estimator.latest_training_job.name

logs_client = boto3.client("logs", region_name=region_name)

response = logs_client.describe_log_streams(
    logGroupName="/aws/sagemaker/TrainingJobs",
    logStreamNamePrefix=training_job_name,
)

log_streams = response.get("logStreams", [])

if log_streams:
    log_stream_name = log_streams[0]["logStreamName"]
    print(f"log_stream_name: {log_stream_name}")
else:
    print("로그 스트림 아직 없음 (학습 시작 전이거나 딜레이)")

print(log_stream_name)

log_stream_name: sean-training-2026-03-06-01-49-19-312/algo-1-1772761799
sean-training-2026-03-06-01-49-19-312/algo-1-1772761799


In [13]:
tail_training_logs(training_job_name, log_stream_name)

2026-03-06 01:51:03,897 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-03-06 01:51:03,898 sagemaker-training-toolkit INFO     Failed to parse hyperparameter dataset_profile_table_name value automl-dataset-profile-experiment-result to Json.
Returning the value itself
2026-03-06 01:51:03,898 sagemaker-training-toolkit INFO     Failed to parse hyperparameter dataset_table_name value automl-dataset to Json.
Returning the value itself
2026-03-06 01:51:03,898 sagemaker-training-toolkit INFO     Failed to parse hyperparameter experiment_hashkey value 1cbd8309 to Json.
Returning the value itself
2026-03-06 01:51:03,898 sagemaker-training-toolkit INFO     Failed to parse hyperparameter job_type value training to Json.
Returning the value itself
2026-03-06 01:51:03,898 sagemaker-training-toolkit INFO     Failed to parse hyperparameter model_experiment_result_table_name value automl-classification-experiment to Json.
Returning the value itself
2026-03-06 0

In [13]:
import json
import os
from datetime import datetime

import boto3

def lambda_handler(event, context):
    region = os.environ.get("AWS_REGION", "us-east-1")
    sm_client = boto3.client("sagemaker", region_name=region)

    sm_uri = event["sm_uri"]
    role_arn = event["role_arn"]
    base_job_name = event.get("base_job_name", "hjsong-lambda-automl-train")

    training_job_name = (
        f"{base_job_name}-{datetime.utcnow().strftime('%Y%m%d-%H%M%S')}"
    )

    hyperparameters = {
        "table_name": "automl-classification-experiment",
        "project_hashkey": "2ee07a49",
        "experiment_hashkey": "1cbd8309",
        "dataset_table_name": "automl-dataset",
        "dataset_profile_table_name": (
            "automl-dataset-profile-experiment-result"
        ),
        "model_repo_table_name": "automl-model-repo",
        "model_experiment_result_table_name": (
            "automl-classification-experiment"
        ),
        "username": "sean@gs.co.kr",
        "job_type": "training",
        "task_token": "1234",
    }

    response = sm_client.create_training_job(
        TrainingJobName=training_job_name,
        AlgorithmSpecification={
            "TrainingImage": sm_uri,
            "TrainingInputMode": "File",
        },
        RoleArn=role_arn,
        OutputDataConfig={
            "S3OutputPath": "s3://retail-mlops-edu-202602/edu-2w/hjsong/output"
        },
        ResourceConfig={
            "InstanceType": "ml.m5.large",
            "InstanceCount": 1,
            "VolumeSizeInGB": 30,
        },
        StoppingCondition={
            "MaxRuntimeInSeconds": 86400
        },
        HyperParameters=hyperparameters,
        Tags=[
            {"Key": "Environment", "Value": "dev"},
            {"Key": "Project", "Value": "automl"},
            {"Key": "Owner", "Value": "sean"},
            {"Key": "CostCenter", "Value": "gs-retail"},
        ],
    )

    return {
        "statusCode": 200,
        "body": json.dumps(
            {
                "message": "Training job submitted",
                "training_job_name": training_job_name,
                "training_job_arn": response["TrainingJobArn"],
            }
        ),
    }

In [14]:
event = {
    "sm_uri" : sm_uri
    ,"role_arn" : role
}
lambda_handler(event=event, context="lambda_test_hjsong")

/tmp/ipykernel_24086/3000961000.py:16: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  f"{base_job_name}-{datetime.utcnow().strftime('%Y%m%d-%H%M%S')}"


{'statusCode': 200,
 'body': '{"message": "Training job submitted", "training_job_name": "hjsong-lambda-automl-train-20260306-065311", "training_job_arn": "arn:aws:sagemaker:us-east-1:155954279556:training-job/hjsong-lambda-automl-train-20260306-065311"}'}